# GQA / MQA — Paper-to-Code Mock (Colab)

**Papers:** GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints (Ainslie et al., 2023) — https://arxiv.org/abs/2305.13245 · Multi-Query Attention (Shazeer, 2019) — https://arxiv.org/abs/1911.02150

Medium mock (~60 min). Fill in the `GroupedQueryAttention` stub, run the content-retrieval demo + KV-cache ratio, then the sanity checks. Reference solution in the last cell. Builds on the attention mock — do that first.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)


def scaled_dot_product_attention(q, k, v, mask=None):
    """q,k,v: (..., T, d_k). Returns (output, attention_weights)."""
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = scores.softmax(dim=-1)
    return attn @ v, attn

## 1. Implement Grouped-Query Attention (YOUR TASK)

Keep `n_heads` query heads but only `n_kv_heads` (=G) key/value heads. Project Q to H heads, K/V to G heads, then `repeat_interleave` the KV heads `H/G` times so each query head attends to its group's shared K, V. `G == n_heads` -> standard MHA; `G == 1` -> MQA.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_model, self.n_heads, self.n_kv_heads = d_model, n_heads, n_kv_heads
        self.d_k = d_model // n_heads
        self.group_size = n_heads // n_kv_heads
        # TODO: wq -> n_heads*d_k ; wk, wv -> n_kv_heads*d_k ; wo -> d_model
        raise NotImplementedError("create wq, wk, wv, wo")

    def _split(self, x, n):                       # (B,T,n*d_k) -> (B,n,T,d_k)
        B, T, _ = x.shape
        return x.view(B, T, n, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        # TODO: q = split(wq(x), n_heads); k,v = split(wk/wv(x), n_kv_heads)
        # TODO: k = k.repeat_interleave(group_size, dim=1); same for v
        # TODO: out, attn = scaled_dot_product_attention(q, k, v, mask)
        # TODO: concat heads -> (B,T,d_model), return wo(out), attn
        raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (content retrieval + KV-cache ratio)
Each sequence has one flagged token (feature 0) whose payload (feature 1) is the target. Train MHA (G=H), GQA (G=2), MQA (G=1) and check they all learn it about equally — then print the exact KV-cache ratio G/H. The cache ratio is exact; quality parity is approximate. The *speed* win needs real-scale serving.

In [ ]:
def make_batch(B, T, d):
    x = torch.randn(B, T, d) * 0.5
    special = torch.randint(0, T, (B,))
    payload = torch.randn(B)
    x[torch.arange(B), special, 0] = 3.0      # flag
    x[torch.arange(B), special, 1] = payload  # payload
    return x, payload.unsqueeze(1), special

def train_model(n_kv_heads, steps=800, seed=0):
    torch.manual_seed(seed)
    B, T, d, H = 256, 6, 16, 4
    gqa, readout = GroupedQueryAttention(d, H, n_kv_heads), nn.Linear(d, 1)
    opt = torch.optim.Adam(list(gqa.parameters()) + list(readout.parameters()), lr=3e-3)
    for step in range(steps):
        x, y, _ = make_batch(B, T, d)
        loss = F.mse_loss(readout(gqa(x)[0].mean(dim=1)), y)
        opt.zero_grad(); loss.backward(); opt.step()
    x, y, special = make_batch(B, T, d)
    attn = gqa(x)[1]
    on_flag = attn.mean(dim=(1, 2))[torch.arange(B), special].mean().item()
    return loss.item(), on_flag

H, chance = 4, 1 / 6
for G in (4, 2, 1):
    loss, flag = train_model(n_kv_heads=G)
    name = {4: "MHA", 2: "GQA", 1: "MQA"}[G]
    print(f"{name} (G={G}): loss {loss:.4f}  attn-on-flag {flag:.2f} "
          f"(chance {chance:.2f})  KV-cache = {G}/{H} = {G/H:.2f} of MHA")

## 3. Sanity checks

In [ ]:
# 1) Q has n_heads, K/V have n_kv_heads; output shape == input
gqa = GroupedQueryAttention(d_model=32, n_heads=8, n_kv_heads=2)
x = torch.randn(4, 10, 32)
q = gqa._split(gqa.wq(x), gqa.n_heads); k = gqa._split(gqa.wk(x), gqa.n_kv_heads)
out, _ = gqa(x)
assert q.shape == (4, 8, 10, 4) and k.shape == (4, 2, 10, 4) and out.shape == x.shape

# 2) repeat_interleave maps each query head to the correct KV group
gqa = GroupedQueryAttention(d_model=16, n_heads=4, n_kv_heads=2)
xb = torch.randn(2, 5, 16); k = gqa._split(gqa.wk(xb), gqa.n_kv_heads)
kr = k.repeat_interleave(gqa.group_size, dim=1)
assert torch.equal(kr[:, 0], k[:, 0]) and torch.equal(kr[:, 1], k[:, 0])
assert torch.equal(kr[:, 2], k[:, 1]) and torch.equal(kr[:, 3], k[:, 1])

# 3) attention rows sum to 1
gqa = GroupedQueryAttention(d_model=32, n_heads=8, n_kv_heads=4)
_, attn = gqa(torch.randn(3, 7, 32))
assert torch.allclose(attn.sum(dim=-1), torch.ones(3, 8, 7), atol=1e-5) and (attn >= 0).all()

# 4) KV-cache element count = G/H of full MHA
B, T, d_model, H, G = 1, 128, 512, 16, 4; d_k = d_model // H
mha_kv, gqa_kv = 2 * B * H * T * d_k, 2 * B * G * T * d_k
assert abs(gqa_kv / mha_kv - G / H) < 1e-9

# 5) n_kv_heads == n_heads reduces to standard MHA
gqa = GroupedQueryAttention(d_model=32, n_heads=4, n_kv_heads=4)
assert gqa.group_size == 1
xb = torch.randn(2, 6, 32); out_gqa, _ = gqa(xb)
qm, km, vm = (gqa._split(p, 4) for p in (gqa.wq(xb), gqa.wk(xb), gqa.wv(xb)))
om, _ = scaled_dot_product_attention(qm, km, vm)
om = gqa.wo(om.transpose(1, 2).contiguous().view(2, 6, 32))
assert torch.allclose(out_gqa, om, atol=1e-6)

# 6) gradients flow to q,k,v,o projections
gqa = GroupedQueryAttention(d_model=32, n_heads=8, n_kv_heads=2)
gqa(torch.randn(4, 7, 32))[0].sum().backward()
for name in ("wq", "wk", "wv", "wo"):
    g = getattr(gqa, name).weight.grad
    assert g is not None and g.abs().sum() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class GroupedQueryAttentionSolution(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.d_model, self.n_heads, self.n_kv_heads = d_model, n_heads, n_kv_heads
        self.d_k = d_model // n_heads
        self.group_size = n_heads // n_kv_heads
        self.wq = nn.Linear(d_model, n_heads * self.d_k)
        self.wk = nn.Linear(d_model, n_kv_heads * self.d_k)
        self.wv = nn.Linear(d_model, n_kv_heads * self.d_k)
        self.wo = nn.Linear(n_heads * self.d_k, d_model)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        q = self._split(self.wq(x), self.n_heads)
        k = self._split(self.wk(x), self.n_kv_heads)
        v = self._split(self.wv(x), self.n_kv_heads)
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        out, attn = scaled_dot_product_attention(q, k, v, mask)
        B, _, T, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.d_k)
        return self.wo(out), attn